##Metrics

In [1]:
from google.colab import drive; drive.mount("/content/drive")

Mounted at /content/drive


##Login to HuggingFace

In [4]:
from huggingface_hub import login
login()

In [3]:
import json
import pandas as pd
import re
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Load results ──────────────────────────────────────────
df = pd.read_csv("/content/drive/MyDrive/AGI_Research/fol_results_Qwen2.5_Llama3.1.csv")
print(f"Loaded {len(df)} results")
print(f"Models: {df['model'].unique()}")

# ── Helpers ───────────────────────────────────────────────

def normalize_fol(formula):
    """Normalize FOL formula for fair comparison."""
    if not isinstance(formula, str):
        return ""
    f = formula.strip()
    f = re.sub(r'\s+', ' ', f)           # collapse whitespace
    f = f.replace('∀', 'forall').replace('∃', 'exists')
    f = f.replace('→', '->').replace('¬', 'NOT').replace('∧', 'AND').replace('∨', 'OR')
    f = re.sub(r'\*+', '', f)            # remove ** artifacts
    f = f.lower()
    return f.strip()

def tokenize_fol(formula):
    """Tokenize FOL formula into meaningful units."""
    f = normalize_fol(formula)
    # Split on spaces and FOL punctuation but keep tokens meaningful
    tokens = re.findall(r'[a-z_]+|\d+|->|<=|>=|!=|[().,∧∨¬→∀∃]', f)
    return tokens

def exact_match(pred, gold):
    """Strict exact match after normalization."""
    return int(normalize_fol(pred) == normalize_fol(gold))

def soft_exact_match(pred, gold):
    """Exact match ignoring parentheses and spacing."""
    def strip_parens(s):
        return re.sub(r'[\s()]+', '', normalize_fol(s))
    return int(strip_parens(pred) == strip_parens(gold))

def bleu_score(pred, gold):
    """Standard BLEU on word-tokenized FOL."""
    pred_tokens  = tokenize_fol(pred)
    gold_tokens  = tokenize_fol(gold)
    if not pred_tokens or not gold_tokens:
        return 0.0
    smoother = SmoothingFunction().method1
    return sentence_bleu([gold_tokens], pred_tokens,
                         weights=(0.5, 0.5, 0, 0),   # BLEU-2
                         smoothing_function=smoother)

def fol_bleu(pred, gold):
    """FOL-aware BLEU: tokenizes on FOL operators and predicate names."""
    def fol_tokenize(f):
        f = normalize_fol(f)
        return re.findall(r'forall|exists|not|and|or|->|[a-z][a-z0-9_]*|[(),.]', f)
    pred_tokens = fol_tokenize(pred)
    gold_tokens = fol_tokenize(gold)
    if not pred_tokens or not gold_tokens:
        return 0.0
    smoother = SmoothingFunction().method1
    return sentence_bleu([gold_tokens], pred_tokens,
                         weights=(0.4, 0.3, 0.2, 0.1),
                         smoothing_function=smoother)

def predicate_overlap(pred, gold):
    """F1 overlap of predicate names used in pred vs gold."""
    def get_predicates(f):
        f = normalize_fol(f)
        return set(re.findall(r'[a-z][a-z0-9_]*(?=\()', f))
    pred_preds = get_predicates(pred)
    gold_preds = get_predicates(gold)
    if not gold_preds:
        return 0.0
    if not pred_preds:
        return 0.0
    tp = len(pred_preds & gold_preds)
    precision = tp / len(pred_preds)
    recall    = tp / len(gold_preds)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def quantifier_match(pred, gold):
    """Check if quantifier structure matches (forall/exists counts)."""
    def count_quantifiers(f):
        f = normalize_fol(f)
        return {
            'forall': len(re.findall(r'\bforall\b', f)),
            'exists': len(re.findall(r'\bexists\b', f)),
        }
    return int(count_quantifiers(pred) == count_quantifiers(gold))

def syntax_validity(formula):
    """
    Basic FOL syntax check:
    - Balanced parentheses
    - Contains at least one predicate (uppercase start or known operator)
    - Not empty
    """
    if not isinstance(formula, str) or not formula.strip():
        return 0
    # Check balanced parentheses
    depth = 0
    for ch in formula:
        if ch == '(': depth += 1
        elif ch == ')': depth -= 1
        if depth < 0:
            return 0
    if depth != 0:
        return 0
    # Must contain at least one predicate-like token
    has_predicate = bool(re.search(r'[A-Za-z][A-Za-z0-9_]*\s*\(', formula))
    return int(has_predicate)

def premise_count_match(pred_premises, gold_premise_count):
    """Check if number of generated premises matches number of NL premises."""
    if not isinstance(pred_premises, str):
        return 0
    # premises stored as pipe-separated in CSV
    count = len([p for p in pred_premises.split('|') if p.strip()])
    return int(count == gold_premise_count)

def predicate_consistency(premises, conclusion):
    """
    Check if predicates used in conclusion are a subset of
    those defined in premises — no new predicates introduced.
    """
    def get_preds(f):
        if not isinstance(f, str): return set()
        return set(re.findall(r'[A-Z][A-Za-z0-9_]*(?=\()', f))

    prem_preds = get_preds(premises)
    conc_preds = get_preds(conclusion)

    if not conc_preds:
        return 0
    # All conclusion predicates should appear in premises
    return int(conc_preds.issubset(prem_preds))

# ── NOTE on gold FOL ──────────────────────────────────────
# FOLIO dataset does not include gold FOL formulas in the public split.
# So BLEU and exact match are computed between models (cross-model agreement)
# rather than against a gold standard. Syntax and structural metrics
# are computed on the generated output directly.
print("\nNote: FOLIO public split has no gold FOL.")
print("BLEU/exact match = cross-model agreement (Llama vs Qwen).")
print("Syntax/structural metrics computed on raw output.\n")

# ── Per-row metrics ───────────────────────────────────────
records = []
for _, row in df.iterrows():
    conclusion = str(row.get('conclusion_fol', ''))
    premises   = str(row.get('premises_fol', ''))
    nl_prem    = str(row.get('nl_premises', ''))
    gold_count = len([p for p in nl_prem.split('|') if p.strip()])

    records.append({
        'model':            row['model'],
        'example_id':       row['example_id'],
        'conclusion_fol':   conclusion,
        'premises_fol':     premises,
        'parse_success':    row.get('parse_success', False),
        'syntax_valid':     syntax_validity(conclusion),
        'premise_count_match': premise_count_match(premises, gold_count),
        'quantifier_ok':    quantifier_match(conclusion, conclusion),  # self — updated below
    })

results_df = pd.DataFrame(records)

# ── Cross-model metrics (Llama vs Qwen) ──────────────────
models = df['model'].unique().tolist()
cross_rows = []

if len(models) >= 2:
    m1_name, m2_name = models[0], models[1]
    m1 = df[df['model'] == m1_name].set_index('example_id')
    m2 = df[df['model'] == m2_name].set_index('example_id')
    shared_ids = set(m1.index) & set(m2.index)

    for eid in sorted(shared_ids):
        pred1 = str(m1.loc[eid, 'conclusion_fol'])
        pred2 = str(m2.loc[eid, 'conclusion_fol'])
        prem1 = str(m1.loc[eid, 'premises_fol'])
        prem2 = str(m2.loc[eid, 'premises_fol'])

        cross_rows.append({
            'example_id':          eid,
            'exact_match':         exact_match(pred1, pred2),
            'soft_exact_match':    soft_exact_match(pred1, pred2),
            'bleu':                bleu_score(pred1, pred2),
            'fol_bleu':            fol_bleu(pred1, pred2),
            'predicate_f1':        predicate_overlap(pred1, pred2),
            'quantifier_match':    quantifier_match(pred1, pred2),
            f'{m1_name}_conclusion': pred1,
            f'{m2_name}_conclusion': pred2,
        })

    cross_df = pd.DataFrame(cross_rows)

# ── Summary report ────────────────────────────────────────
print("=" * 60)
print(" PER-MODEL METRICS")
print("=" * 60)
summary = results_df.groupby('model').agg(
    examples        = ('example_id',        'count'),
    parse_success   = ('parse_success',      'mean'),
    syntax_valid    = ('syntax_valid',       'mean'),
    premise_match   = ('premise_count_match','mean'),
).reset_index()
summary['parse_success'] = (summary['parse_success'] * 100).round(1).astype(str) + '%'
summary['syntax_valid']  = (summary['syntax_valid']  * 100).round(1).astype(str) + '%'
summary['premise_match'] = (summary['premise_match'] * 100).round(1).astype(str) + '%'
print(summary.to_string(index=False))

if len(models) >= 2:
    print(f"\n{'='*60}")
    print(f" CROSS-MODEL AGREEMENT ({m1_name} vs {m2_name})")
    print(f"{'='*60}")
    print(f"  Exact Match          : {cross_df['exact_match'].mean()*100:.1f}%")
    print(f"  Soft Exact Match     : {cross_df['soft_exact_match'].mean()*100:.1f}%")
    print(f"  BLEU-2               : {cross_df['bleu'].mean():.4f}")
    print(f"  FOL-BLEU             : {cross_df['fol_bleu'].mean():.4f}")
    print(f"  Predicate F1         : {cross_df['predicate_f1'].mean():.4f}")
    print(f"  Quantifier Match     : {cross_df['quantifier_match'].mean()*100:.1f}%")

    print(f"\n{'='*60}")
    print(f" PER-EXAMPLE BREAKDOWN")
    print(f"{'='*60}")
    print(cross_df[['example_id','exact_match','soft_exact_match',
                    'bleu','fol_bleu','predicate_f1','quantifier_match']].to_string(index=False))

# ── Save metrics ──────────────────────────────────────────
results_df.to_csv("/content/drive/MyDrive/AGI_Research/per_model_metrics_all.csv", index=False)
if len(models) >= 2:
    cross_df.to_csv("/content/drive/MyDrive/AGI_Research/cross_model_metrics_all.csv", index=False)
print("\nMetrics saved to Drive.")

Loaded 406 results
Models: ['Llama-3.1-8B-Instruct' 'Qwen2.5-7B-Instruct']

Note: FOLIO public split has no gold FOL.
BLEU/exact match = cross-model agreement (Llama vs Qwen).
Syntax/structural metrics computed on raw output.

 PER-MODEL METRICS
                model  examples parse_success syntax_valid premise_match
Llama-3.1-8B-Instruct       203         12.8%        61.6%         12.8%
  Qwen2.5-7B-Instruct       203        100.0%       100.0%         83.7%

 CROSS-MODEL AGREEMENT (Llama-3.1-8B-Instruct vs Qwen2.5-7B-Instruct)
  Exact Match          : 6.9%
  Soft Exact Match     : 10.8%
  BLEU-2               : 0.3161
  FOL-BLEU             : 0.2565
  Predicate F1         : 0.2379
  Quantifier Match     : 100.0%

 PER-EXAMPLE BREAKDOWN
 example_id  exact_match  soft_exact_match     bleu  fol_bleu  predicate_f1  quantifier_match
          0            0                 0 0.000000  0.000000      0.000000                 1
          1            0                 0 0.000000  0.000000  

In [5]:
from datasets import load_dataset
import pandas as pd

# Load validation split
folio = load_dataset("yale-nlp/folio", split="validation")

# Read model outputs
df = pd.read_csv("/content/drive/MyDrive/AGI_Research/fol_results_Qwen2.5_Llama3.1.csv")

# Add gold FOL using example_id
df["gold_premises_fol"] = df["example_id"].apply(
    lambda i: folio[int(i)]["premises-FOL"]
)

df["gold_conclusion_fol"] = df["example_id"].apply(
    lambda i: folio[int(i)]["conclusion-FOL"]
)

df.to_csv("/content/drive/MyDrive/AGI_Research/fol_results_with_gold_all.csv",
          index=False)

print(df.columns)

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

folio_v2_train.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

folio_v2_validation.jsonl:   0%|          | 0.00/208k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Index(['model', 'example_id', 'nl_premises', 'nl_conclusion', 'gold_label',
       'entities', 'premises_fol', 'conclusion_fol', 'parse_success',
       'gold_premises_fol', 'gold_conclusion_fol'],
      dtype='object')


In [6]:
results = []

for _, row in df.iterrows():

    results.append({

        "model": row["model"],
        "example_id": row["example_id"],

        "premise_em":
            exact_match(row["premises_fol"],
                        row["gold_premises_fol"]),

        "conclusion_em":
            exact_match(row["conclusion_fol"],
                        row["gold_conclusion_fol"]),

        "premise_bleu":
            bleu_score(row["premises_fol"],
                       row["gold_premises_fol"]),

        "conclusion_bleu":
            bleu_score(row["conclusion_fol"],
                       row["gold_conclusion_fol"]),

        "premise_fol_bleu":
            fol_bleu(row["premises_fol"],
                     row["gold_premises_fol"]),

        "conclusion_fol_bleu":
            fol_bleu(row["conclusion_fol"],
                     row["gold_conclusion_fol"]),

        "premise_predicate_f1":
            predicate_overlap(row["premises_fol"],
                              row["gold_premises_fol"]),

        "conclusion_predicate_f1":
            predicate_overlap(row["conclusion_fol"],
                              row["gold_conclusion_fol"]),

        "premise_quantifier":
            quantifier_match(row["premises_fol"],
                             row["gold_premises_fol"]),

        "conclusion_quantifier":
            quantifier_match(row["conclusion_fol"],
                             row["gold_conclusion_fol"]),

        "premise_tree":
            syntax_validity(row["premises_fol"]),

        "conclusion_tree":
            syntax_validity(row["conclusion_fol"])
    })

metrics_df = pd.DataFrame(results)

In [7]:
summary = metrics_df.groupby("model").mean(numeric_only=True)

summary = summary.round(3)

print(summary)

                       example_id  premise_em  conclusion_em  premise_bleu  \
model                                                                        
Llama-3.1-8B-Instruct       101.0         0.0          0.049         0.052   
Qwen2.5-7B-Instruct         101.0         0.0          0.113         0.433   

                       conclusion_bleu  premise_fol_bleu  conclusion_fol_bleu  \
model                                                                           
Llama-3.1-8B-Instruct            0.228             0.043                0.176   
Qwen2.5-7B-Instruct              0.428             0.364                0.348   

                       premise_predicate_f1  conclusion_predicate_f1  \
model                                                                  
Llama-3.1-8B-Instruct                 0.044                    0.187   
Qwen2.5-7B-Instruct                   0.333                    0.292   

                       premise_quantifier  conclusion_quantifier  \
model

In [8]:
import pandas as pd
import re

df = pd.read_csv("/content/drive/MyDrive/AGI_Research/fol_results_Qwen2.5_Llama3.1.csv")

# ── Normalizer ────────────────────────────────────────────
def normalize(f):
    if not isinstance(f, str): return ""
    f = f.strip().rstrip('.')
    f = re.sub(r'\*+', '', f)           # remove ** artifacts
    f = re.sub(r'###.*', '', f)         # remove ### artifacts
    f = re.sub(r'\|?\s*Note:.*', '', f) # remove inline notes
    f = f.replace('∀', 'forall').replace('∃', 'exists')
    f = f.replace('→', '->').replace('↔', '<->').replace('∼', 'NOT')
    f = f.replace('¬', 'NOT ').replace('∧', 'AND').replace('∨', 'OR')
    f = re.sub(r'\s+', ' ', f)
    return f.strip()

# ── Syntax checks ─────────────────────────────────────────

def balanced_parens(f):
    """Parentheses are balanced."""
    depth = 0
    for ch in f:
        if ch == '(': depth += 1
        elif ch == ')': depth -= 1
        if depth < 0: return 0
    return int(depth == 0)

def has_predicate(f):
    """At least one predicate call like Foo(x)."""
    return int(bool(re.search(r'[A-Za-z][A-Za-z0-9_]*\s*\(', f)))

def has_quantifier(f):
    """Contains at least one quantifier."""
    return int(bool(re.search(r'\b(forall|exists)\b', f, re.IGNORECASE)))

def no_artifacts(f):
    """No leftover ### or ** or Note: in the formula."""
    return int(not bool(re.search(r'(###|\*\*|Note:)', f)))

def valid_operators_only(f):
    """Uses recognized FOL operators — no stray natural language."""
    # Strip known good tokens and check nothing alarming remains
    cleaned = re.sub(r'\b(forall|exists|AND|OR|NOT|->|<->)\b', '', f, flags=re.IGNORECASE)
    cleaned = re.sub(r'[A-Za-z][A-Za-z0-9_]*\s*\([^)]*\)', '', cleaned)  # predicates
    cleaned = re.sub(r'[A-Za-z][A-Za-z0-9_]*', '', cleaned)               # constants/vars
    cleaned = re.sub(r'[\s(),.\d=]', '', cleaned)
    # anything left that isn't a known symbol is suspicious
    suspicious = re.sub(r'[-><!]', '', cleaned)
    return int(len(suspicious) == 0)

def conclusion_grounded(conclusion, entities):
    """
    Check if conclusion references a named entity (constant)
    rather than only anonymous variables.
    """
    if not isinstance(entities, str) or not entities.strip():
        return 0
    entity_list = [e.strip() for e in entities.split(',') if e.strip()]
    for entity in entity_list:
        if entity and entity in conclusion:
            return 1
    return 0

def premise_artifact_free(premises):
    """No inline commentary in premises."""
    if not isinstance(premises, str): return 0
    return int(not bool(re.search(r'(Note:|###|\*\*)', premises)))

# ── Apply ─────────────────────────────────────────────────
rows = []
for _, row in df.iterrows():
    conc  = normalize(str(row.get('conclusion_fol', '')))
    prems = str(row.get('premises_fol', ''))
    ents  = str(row.get('entities', ''))

    rows.append({
        'model':              row['model'],
        'example_id':         row['example_id'],
        'parse_success':      int(str(row.get('parse_success','')).strip() == 'True'),
        'balanced_parens':    balanced_parens(conc),
        'has_predicate':      has_predicate(conc),
        'has_quantifier':     has_quantifier(conc),
        'no_artifacts':       no_artifacts(conc),
        'valid_operators':    valid_operators_only(conc),
        'conclusion_grounded':conclusion_grounded(conc, ents),
        'premise_artifact_free': premise_artifact_free(prems),
        'predicate_consistency': predicate_consistency(prems, conc),
    })

syntax_df = pd.DataFrame(rows)

# ── Composite syntax score ────────────────────────────────
checks = ['balanced_parens','has_predicate','no_artifacts',
          'valid_operators','conclusion_grounded','premise_artifact_free', 'predicate_consistency']
syntax_df['syntax_score'] = syntax_df[checks].mean(axis=1).round(3)

# ── Report ────────────────────────────────────────────────
print("=" * 65)
print(" SYNTAX METRICS PER MODEL")
print("=" * 65)
summary = syntax_df.groupby('model')[
    ['parse_success','balanced_parens','has_predicate','has_quantifier',
     'no_artifacts','valid_operators','conclusion_grounded',
     'premise_artifact_free','syntax_score']
].mean().round(3) * 100

print(summary.T.to_string())

print("\n")
print("=" * 65)
print(" PER-EXAMPLE SYNTAX BREAKDOWN")
print("=" * 65)
print(syntax_df[['model','example_id'] + checks + ['syntax_score']].to_string(index=False))

syntax_df.to_csv("/content/drive/MyDrive/AGI_Research/syntax_metrics_all.csv", index=False)
print("\nSaved: syntax_metrics_all.csv")


 SYNTAX METRICS PER MODEL
model                  Llama-3.1-8B-Instruct  Qwen2.5-7B-Instruct
parse_success                           12.8                100.0
balanced_parens                         99.5                100.0
has_predicate                           62.1                100.0
has_quantifier                           0.0                  0.0
no_artifacts                           100.0                100.0
valid_operators                         97.0                 97.0
conclusion_grounded                     37.4                 67.0
premise_artifact_free                  100.0                100.0
syntax_score                            72.1                 92.6


 PER-EXAMPLE SYNTAX BREAKDOWN
                model  example_id  balanced_parens  has_predicate  no_artifacts  valid_operators  conclusion_grounded  premise_artifact_free  predicate_consistency  syntax_score
Llama-3.1-8B-Instruct           0                1              0             1                1        

In [9]:
EXAMPLE = 0

gold = folio[EXAMPLE]

print("="*80)
print(f"Example {EXAMPLE}")
print("="*80)

print("\nNatural Language Premises\n")
print(gold["premises"])

print("\nGold Premises FOL\n")
print(gold["premises-FOL"])

print("\nGold Conclusion\n")
print(gold["conclusion-FOL"])

print("\n" + "="*80)

for model in df["model"].unique():

    row = df[
        (df.model == model) &
        (df.example_id == EXAMPLE)
    ].iloc[0]

    print(f"\nMODEL: {model}")

    print("\nPremises FOL")
    print(row["premises_fol"])

    print("\nConclusion FOL")
    print(row["conclusion_fol"])

    print("\nPremise BLEU:",
          bleu_score(row["premises_fol"],
                     gold["premises-FOL"]))

    print("Conclusion BLEU:",
          bleu_score(row["conclusion_fol"],
                     gold["conclusion-FOL"]))

    print("-"*80)

Example 0

Natural Language Premises

People in this club who perform in school talent shows often attend and are very engaged with school events.
People in this club either perform in school talent shows often or are inactive and disinterested community members.
People in this club who chaperone high school dances are not students who attend the school.
All people in this club who are inactive and disinterested members of their community chaperone high school dances.
All young children and teenagers in this club who wish to further their academic careers and educational opportunities are students who attend the school. 
Bonnie is in this club and she either both attends and is very engaged with school events and is a student who attends the school or is not someone who both attends and is very engaged with school events and is not a student who attends the school.

Gold Premises FOL

∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, 